# Neural Identifier Training with Particle Filters - Van der Pol Oscillator

In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
# ============================================================
# 1) True nonlinear system (Van der Pol Oscillator)
# ============================================================
def plant_dynamics(x, u, mu=1.0):
    """
    Continuous dynamics for Van der Pol oscillator: x = [x1, x2]. 
    Returns x_dot.
    
    The Van der Pol equations:
    dx1/dt = x2
    dx2/dt = μ(1 - x1²)x2 - x1
    """
    x1, x2 = x
    
    # Van der Pol equations
    x1_dot = x2
    x2_dot = mu * (1 - x1**2) * x2 - x1
    
    return np.array([x1_dot, x2_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

In [3]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est):
    """
    Features for a 2-state Van der Pol oscillator (no inputs):
    z = [S(x1), S(x2), S(x1)S(x2), S(x1)^2, S(x2)^2, S(x1)^3, x1, x2, 1]
    """
    s_x1 = sigmoidal(x_est[0])  # x1 (position)
    s_x2 = sigmoidal(x_est[1])  # x2 (velocity)
    
    return np.array([
        # s_x1, s_x2,                           # Sigmoid terms
        s_x1*s_x2,                            # Cross term
        s_x1**2, s_x2**2,                     # Quadratic sigmoid terms
        s_x1**3,                              # Cubic term (important for Van der Pol)
        # x_est[0], x_est[1],                   # Linear terms (direct states)
        # 1.0                                    # Bias
    ])


def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [4]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x1 (position - measured output for Van der Pol oscillator)

        z_i = construct_z_vector(x_state_for_z)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

In [5]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        # Vectorized particle initialization
        if initial_weights is not None:
            # Stack initial weights and add noise to all particles at once
            base_weights = np.array([initial_weights[i] if i < len(initial_weights) 
                        else np.random.randn(num_weights_per_neuron) * 0.1 
                        for i in range(num_neurons)])  # (num_neurons, num_weights_per_neuron)
            
            # Generate all particles for all neurons at once
            noise = np.random.randn(num_neurons, n_particles, num_weights_per_neuron) * 0.1
            particles_all = base_weights[:, np.newaxis, :] + noise  # (num_neurons, n_particles, num_weights_per_neuron)
        else:
            # Generate all particles from scratch
            particles_all = np.random.randn(num_neurons, n_particles, num_weights_per_neuron) * 0.1
        
        # Convert to list of arrays for compatibility with existing code
        self.particles = [particles_all[i] for i in range(num_neurons)]
        self.weights_pf = [np.ones(n_particles) / n_particles for _ in range(num_neurons)]

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]
        
        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.searchsorted(cdf, u0 + np.arange(N) / N)

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x (measured output for Lorenz system)
        z = construct_z_vector(x_state_for_z)  # (num_features,)

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]


In [6]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        # Calculate lambda and weights for sigma points
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Weights for mean and covariance computation
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        """Generate sigma points for UKF."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x (measured output for Lorenz system)

        z_i = construct_z_vector(x_state_for_z)  # shape (num_features,)

        for i in range(self.num_neurons):
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

In [7]:
# ============================================================
# 5) Parameter Optimization (Optional)
# ============================================================
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

def run_simulation_with_params(params, filter_type='UKF', n_steps=500, verbose=False):
    """
    Run a simulation with given parameters and return MSE.
    
    Parameters:
    -----------
    params : array-like
        For UKF: [log10(Q_init), log10(R_init), log10(P_init), eta, log10(alpha)]
        For PF: [log10(Q_std), log10(R_std), n_particles_ratio]
        For EKF: [log10(Q_init), log10(R_init), log10(P_init), eta]
    filter_type : str
        'UKF', 'PF', or 'EKF'
    n_steps : int
        Number of simulation steps (reduced for optimization speed)
    verbose : bool
        Print progress
    
    Returns:
    --------
    float : Total MSE (lower is better)
    """
    dt = 0.01
    process_noise_type = 'gaussian'
    process_noise_std = 0.01
    
    # Initialize true system
    x_true = np.zeros((n_steps, 2))
    x_true[0] = [2.0, 0.0]
    u = 0.0
    
    # RHONN config
    num_neurons = 2
    num_features = 4
    num_weights_per_neuron = num_features
    
    # Fixed initial weights
    np.random.seed(7517)
    common_initial_weights = [np.random.uniform(-1.0, 1.0, num_weights_per_neuron) 
                             for _ in range(num_neurons)]
    
    try:
        if filter_type == 'UKF':
            # Unpack parameters (log scale for most)
            Q_init = 10 ** params[0]
            R_init = 10 ** params[1]
            P_init = 10 ** params[2]
            eta = params[3]
            alpha = 10 ** params[4]
            
            trainer = UKF_RHONN_Trainer(
                num_neurons, num_weights_per_neuron,
                initial_weights=common_initial_weights,
                Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta,
                alpha=alpha, beta=2.0
            )
            x_hat = np.zeros((n_steps, 2))
            x_hat[0] = x_true[0]
            
        elif filter_type == 'PF':
            # Unpack parameters
            Q_std = 10 ** params[0]
            R_std = 10 ** params[1]
            n_particles_ratio = params[2]
            n_particles = int(100 * n_particles_ratio)
            
            trainer = PF_RHONN_Trainer(
                num_neurons, num_weights_per_neuron,
                n_particles=n_particles,
                initial_weights=common_initial_weights,
                Q_std=Q_std, R_std=R_std, ess_threshold=n_particles / 2
            )
            
            # Initialize PF
            for i in range(trainer.num_neurons):
                trainer.particles[i] = np.tile(
                    common_initial_weights[i], (n_particles, 1)
                )
                trainer.weights_pf[i] = np.ones(n_particles) / n_particles
            
            x_hat = np.zeros((n_steps, 2))
            x_hat[0] = x_true[0]
            
        elif filter_type == 'EKF':
            # Unpack parameters
            Q_init = 10 ** params[0]
            R_init = 10 ** params[1]
            P_init = 10 ** params[2]
            eta = params[3]
            
            trainer = EKF_RHONN_Trainer(
                num_neurons, num_weights_per_neuron,
                initial_weights=common_initial_weights,
                Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta
            )
            x_hat = np.zeros((n_steps, 2))
            x_hat[0] = x_true[0]
        
        else:
            raise ValueError(f"Unknown filter type: {filter_type}")
        
        # Run simulation
        for k in range(n_steps - 1):
            # True system step
            x_true[k+1] = plant(x_true[k], u, dt, process_noise_type, process_noise_std)
            
            # Filter update
            trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat[k])
            
            # Prediction
            x_state_for_z = np.copy(x_hat[k])
            x_state_for_z[0] = x_true[k][0]
            
            if filter_type == 'PF':
                weight_estimates = trainer.get_estimate()
                x_hat[k+1, 0] = RHONN_predict(x_state_for_z, weight_estimates[0])
                x_hat[k+1, 1] = RHONN_predict(x_state_for_z, weight_estimates[1])
            else:
                x_hat[k+1, 0] = RHONN_predict(x_state_for_z, trainer.weights[0])
                x_hat[k+1, 1] = RHONN_predict(x_state_for_z, trainer.weights[1])
        
        # Calculate MSE
        mse_x1 = np.mean((x_true[:, 0] - x_hat[:, 0])**2)
        mse_x2 = np.mean((x_true[:, 1] - x_hat[:, 1])**2)
        total_mse = mse_x1 + mse_x2
        
        if verbose:
            print(f"  MSE: {total_mse:.6e} | Params: {params}")
        
        return total_mse
    
    except Exception as e:
        if verbose:
            print(f"  Error with params {params}: {e}")
        return 1e10  # Return large value on error


def optimize_filter_parameters(filter_type='UKF', method='differential_evolution', 
                               n_steps=500, maxiter=50, verbose=True):
    """
    Optimize filter parameters using scipy optimization.
    
    Parameters:
    -----------
    filter_type : str
        'UKF', 'PF', or 'EKF'
    method : str
        'differential_evolution' (global) or 'nelder-mead' (local)
    n_steps : int
        Number of simulation steps for each evaluation
    maxiter : int
        Maximum iterations for optimization
    verbose : bool
        Print progress
    
    Returns:
    --------
    dict : Optimized parameters and final MSE
    """
    print(f"\n{'='*70}")
    print(f"🔧 OPTIMIZING {filter_type} PARAMETERS")
    print(f"{'='*70}")
    print(f"Method: {method}")
    print(f"Simulation length: {n_steps} steps")
    print(f"Max iterations: {maxiter}\n")
    
    # Define parameter bounds and initial guesses
    if filter_type == 'UKF':
        # [log10(Q_init), log10(R_init), log10(P_init), eta, log10(alpha)]
        bounds = [(-6, -2), (-4, -1), (-1, 1), (0.1, 2.0), (-4, -1)]
        x0 = [-5, -2, 0, 1.0, -3]  # log10(1e-5), log10(1e-2), log10(1.0), 1.0, log10(1e-3)
        param_names = ['log10(Q_init)', 'log10(R_init)', 'log10(P_init)', 'eta', 'log10(alpha)']
        
    elif filter_type == 'PF':
        # [log10(Q_std), log10(R_std), n_particles_ratio]
        bounds = [(-2, 1), (-4, -2), (1, 10)]
        x0 = [0, -3, 8]  # log10(1.0), log10(0.001), 800 particles
        param_names = ['log10(Q_std)', 'log10(R_std)', 'n_particles_ratio']
        
    elif filter_type == 'EKF':
        # [log10(Q_init), log10(R_init), log10(P_init), eta]
        bounds = [(-6, -2), (-4, -1), (-1, 1), (0.1, 2.0)]
        x0 = [-3, -2, 0, 0.5]  # log10(1e-3), log10(1e-2), log10(1.0), 0.5
        param_names = ['log10(Q_init)', 'log10(R_init)', 'log10(P_init)', 'eta']
    else:
        raise ValueError(f"Unknown filter type: {filter_type}")
    
    # Objective function
    objective = lambda params: run_simulation_with_params(params, filter_type, n_steps, verbose=False)
    
    # Optimize
    if method == 'differential_evolution':
        result = differential_evolution(
            objective, 
            bounds, 
            maxiter=maxiter,
            popsize=15,
            seed=42,
            atol=1e-6,
            tol=1e-6,
            workers=1,
            updating='deferred',
            disp=verbose
        )
    elif method == 'nelder-mead':
        result = minimize(
            objective,
            x0,
            method='Nelder-Mead',
            options={'maxiter': maxiter, 'disp': verbose, 'xatol': 1e-6, 'fatol': 1e-6}
        )
    else:
        raise ValueError(f"Unknown optimization method: {method}")
    
    # Extract and display results
    optimized_params = result.x
    final_mse = result.fun
    
    print(f"\n{'='*70}")
    print(f"✅ OPTIMIZATION COMPLETE")
    print(f"{'='*70}")
    print(f"Final MSE: {final_mse:.6e}")
    print(f"\nOptimized Parameters:")
    
    if filter_type == 'UKF':
        Q_init = 10 ** optimized_params[0]
        R_init = 10 ** optimized_params[1]
        P_init = 10 ** optimized_params[2]
        eta = optimized_params[3]
        alpha = 10 ** optimized_params[4]
        
        print(f"  Q_init = {Q_init:.6e}")
        print(f"  R_init = {R_init:.6e}")
        print(f"  P_init = {P_init:.6e}")
        print(f"  eta = {eta:.4f}")
        print(f"  alpha = {alpha:.6e}")
        
        return {
            'Q_init': Q_init, 'R_init': R_init, 'P_init': P_init,
            'eta': eta, 'alpha': alpha, 'beta': 2.0,
            'mse': final_mse
        }
        
    elif filter_type == 'PF':
        Q_std = 10 ** optimized_params[0]
        R_std = 10 ** optimized_params[1]
        n_particles = int(100 * optimized_params[2])
        
        print(f"  Q_std = {Q_std:.6e}")
        print(f"  R_std = {R_std:.6e}")
        print(f"  n_particles = {n_particles}")
        
        return {
            'Q_std': Q_std, 'R_std': R_std,
            'n_particles': n_particles,
            'mse': final_mse
        }
        
    elif filter_type == 'EKF':
        Q_init = 10 ** optimized_params[0]
        R_init = 10 ** optimized_params[1]
        P_init = 10 ** optimized_params[2]
        eta = optimized_params[3]
        
        print(f"  Q_init = {Q_init:.6e}")
        print(f"  R_init = {R_init:.6e}")
        print(f"  P_init = {P_init:.6e}")
        print(f"  eta = {eta:.4f}")
        
        return {
            'Q_init': Q_init, 'R_init': R_init, 'P_init': P_init,
            'eta': eta, 'mse': final_mse
        }


# Example usage (uncomment to run optimization):
# optimized_ukf_params = optimize_filter_parameters('UKF', method='differential_evolution', n_steps=500, maxiter=30)
# optimized_pf_params = optimize_filter_parameters('PF', method='differential_evolution', n_steps=500, maxiter=20)
# optimized_ekf_params = optimize_filter_parameters('EKF', method='differential_evolution', n_steps=500, maxiter=30)

## 🔧 Parameter Optimization (Optional)

Uncomment and run the cells below to automatically optimize filter parameters.

**Warning:** Optimization can take several minutes depending on `maxiter` and `n_steps`.

**Methods:**
- `differential_evolution`: Global optimization (recommended, slower but more thorough)
- `nelder-mead`: Local optimization (faster but may find local minimum)

**Parameters to optimize:**
- **UKF**: Q_init, R_init, P_init, eta, alpha
- **PF**: Q_std, R_std, n_particles
- **EKF**: Q_init, R_init, P_init, eta

In [8]:
# # Optimize UKF parameters
# optimized_ukf_params = optimize_filter_parameters(
#     filter_type='UKF', 
#     method='differential_evolution', 
#     n_steps=500,  # Reduced for speed, increase to 1000 for better accuracy
#     maxiter=30,   # Increase for more thorough search
#     verbose=True
# )

In [9]:
# # Optimize PF parameters
# optimized_pf_params = optimize_filter_parameters(
#     filter_type='PF', 
#     method='differential_evolution', 
#     n_steps=500, 
#     maxiter=20,
#     verbose=True
# )

In [10]:
# Optimize EKF parameters (if using EKF)
# optimized_ekf_params = optimize_filter_parameters(
#     filter_type='EKF', 
#     method='differential_evolution', 
#     n_steps=500, 
#     maxiter=30,
#     verbose=True
# )

### How to use optimized parameters:

After running optimization, use the returned dictionary to configure your filters:

```python
# Example: Use optimized UKF parameters
ukf_trainer = UKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=optimized_ukf_params['Q_init'],
    R_init=optimized_ukf_params['R_init'],
    P_init=optimized_ukf_params['P_init'],
    eta=optimized_ukf_params['eta'],
    alpha=optimized_ukf_params['alpha'],
    beta=optimized_ukf_params['beta']
)

# Example: Use optimized PF parameters
pf_trainer = PF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    n_particles=optimized_pf_params['n_particles'],
    initial_weights=common_initial_weights,
    Q_std=optimized_pf_params['Q_std'],
    R_std=optimized_pf_params['R_std'],
    ess_threshold=optimized_pf_params['n_particles'] / 2
)
```

In [11]:
# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'gaussian'  # 'laplacian' | 'uniform' | 'gaussian'
    process_noise_std = 0.01

    measurement_noise_std = 0.05

    # --- True system init ---
    x_true = np.zeros((n_steps, 2))
    x_true[0] = [2.0, 0.0]  # Initial conditions for Van der Pol oscillator [position, velocity]
    u = 0.0

    # --- RHONN config ---
    num_neurons = 2  # Two states for Van der Pol oscillator
    num_features = 4  # Updated feature vector size for 2 states
    num_weights_per_neuron = num_features

    # --- Common initial weights for fair comparison ---
    np.random.seed(7517)  # (optional) reproducibility of initial weights
    common_initial_weights = [np.random.uniform(-1.0, 1.0, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i}: {w}")

    # # --- EKF --- (Tuned parameters for Van der Pol oscillator)
    # ekf_trainer = EKF_RHONN_Trainer(
    #     num_neurons, num_weights_per_neuron,
    #     initial_weights=common_initial_weights,
    #     Q_init=1e-3, R_init=1e-2, P_init=1.0, eta=0.5
    # )
    # x_hat_ekf = np.zeros((n_steps, 2))
    # x_hat_ekf[0] = x_true[0]

    # --- UKF ---
    ukf_trainer = UKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1.0e-2, R_init=1.0e-4, P_init=2.121261, eta=1.0436,
        alpha=2.897616e-4, beta=2.0  # UKF-specific parameters
    )
    x_hat_ukf = np.zeros((n_steps, 2))
    x_hat_ukf[0] = x_true[0]

    # --- PF ---
    n_particles = 980
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        # Q_std=9.243476e-1, R_std=1.766588e-3, ess_threshold=n_particles / 2  # ESS < N/2
        Q_std=1.0, R_std=0.002, ess_threshold=n_particles / 2  # ESS < N/2
        )

    # Force identical particle initialization if desired:
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    x_hat_pf = np.zeros((n_steps, 2))
    x_hat_pf[0] = x_true[0]

    print("Starting simulation...")
    for k in range(n_steps - 1):
        # ---- 1) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u, dt, process_noise_type, process_noise_std) + np.random.randn(num_neurons) * measurement_noise_std

        

        # # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        # ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ekf[k])

        # x_state_for_z_ekf = np.copy(x_hat_ekf[k])
        # x_state_for_z_ekf[0] = x_true[k][0]  # series-parallel uses measured x1 at k
        # x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0])  # x1
        # x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1])  # x2

        # ---- 2b) UKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ukf[k])

        x_state_for_z_ukf = np.copy(x_hat_ukf[k])
        x_state_for_z_ukf[0] = x_true[k][0]  # series-parallel uses measured x1 at k
        x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0])  # x1
        x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1])  # x2

        # ---- 3) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k])

        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0]  # series-parallel uses measured x1 at k
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0])   # x1
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1])   # x2

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")


Common Initial Weights:
  Neuron 0: [ 0.18387164 -0.7100519   0.31646359 -0.55559831]
  Neuron 1: [ 0.98591761 -0.66891831 -0.57953188  0.77237819]
Starting simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 60.0%
Simulation progress: 70.0%
Simulation progress: 80.0%
Simulation progress: 90.0%
Simulation finished.


In [12]:
# ============================================================
# 6) Resultados y gráficas para Oscilador de Van der Pol
# ============================================================

# Configuración de formato para tesis
thesis_config = {
    'font_family': 'Computer Modern, serif',
    'font_size': 14,
    'title_font_size': 16,
    'legend_font_size': 12,
    'line_width_true': 2.5,
    'line_width_est': 2.0,
    'plot_width': 1000,
    'plot_height': 500,
    'grid_color': 'rgba(200, 200, 200, 0.3)',
    'grid_width': 0.5
}

# Cálculo de MSE
# mse_x1_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
# mse_x2_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
mse_x1_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
mse_x2_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
mse_x1_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
mse_x2_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)

# mse_total_ekf = mse_x1_ekf + mse_x2_ekf
mse_total_ukf = mse_x1_ukf + mse_x2_ukf
mse_total_pf = mse_x1_pf + mse_x2_pf

print("="*70)
print(f"🏆 MEJOR FILTRO: ", end="")
mse_dict = {'UKF': mse_total_ukf, 'PF': mse_total_pf}  # EKF commented out
best_filter = min(mse_dict, key=mse_dict.get)
print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.6f})")
print("="*70)

# print(f"\nPesos Finales EKF-RHONN:")
# for i in range(2):
#     print(f"  Neurona {i+1} (x{i+1}): {ekf_trainer.weights[i]}")

print(f"\nPesos Finales UKF-RHONN:")
for i in range(2):
    print(f"  Neurona {i+1} (x{i+1}): {ukf_trainer.weights[i]}")

print(f"\nEstimación de Pesos PF-RHONN:")
pf_estimates = pf_trainer.get_estimate()
for i in range(2):
    print(f"  Neurona {i+1} (x{i+1}): {pf_estimates[i]}")

print("\n--- Comparación de Desempeño (MSE) - Oscilador de Van der Pol ---")
# print(f"EKF MSE x₁ (posición):  {mse_x1_ekf:.6f}")
# print(f"EKF MSE x₂ (velocidad): {mse_x2_ekf:.6f}")
print(f"UKF MSE x₁ (posición):  {mse_x1_ukf:.6f}")
print(f"UKF MSE x₂ (velocidad): {mse_x2_ukf:.6f}")
print(f"PF  MSE x₁ (posición):  {mse_x1_pf:.6f}")
print(f"PF  MSE x₂ (velocidad): {mse_x2_pf:.6f}")


🏆 MEJOR FILTRO: PF (MSE total: 0.000004)

Pesos Finales UKF-RHONN:
  Neurona 1 (x1): [-8.01198716 -5.57437963  3.3841068   7.51976574]
  Neurona 2 (x2): [ 3.30092955 -6.1456971   1.34250718  3.87257485]

Estimación de Pesos PF-RHONN:
  Neurona 1 (x1): [  6.27128305   1.49226857  -2.87541676 -19.75999245]
  Neurona 2 (x2): [ 8.61761097 -5.6309075  -1.20324748  2.5221481 ]

--- Comparación de Desempeño (MSE) - Oscilador de Van der Pol ---
UKF MSE x₁ (posición):  0.000059
UKF MSE x₂ (velocidad): 0.000050
PF  MSE x₁ (posición):  0.000002
PF  MSE x₂ (velocidad): 0.000003


In [13]:

# Gráficas individuales por estado
states_info = [
    {'idx': 0, 'var': 'x₁', 'desc': 'Posición', 'y_label': 'Posición x₁'},
    {'idx': 1, 'var': 'x₂', 'desc': 'Velocidad', 'y_label': 'Velocidad x₂'}
]

for state_info in states_info:
    i = state_info['idx']
    
    fig = go.Figure()
    
    # Estado real (línea negra gruesa)
    fig.add_trace(go.Scatter(
        x=t_history, y=x_true[:, i],
        mode='lines',
        name='Estado Real',
        line=dict(color='#000000', width=thesis_config['line_width_true']),
        showlegend=True
    ))
    
    # # Estimación EKF
    # fig.add_trace(go.Scatter(
    #     x=t_history, y=x_hat_ekf[:, i],
    #     mode='lines',
    #     name='EKF-RHONN',
    #     line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
    #     showlegend=True
    # ))
    
    # Estimación UKF
    fig.add_trace(go.Scatter(
        x=t_history, y=x_hat_ukf[:, i],
        mode='lines',
        name='UKF-RHONN',
        line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
        showlegend=True
    ))
    
    # Estimación PF
    fig.add_trace(go.Scatter(
        x=t_history, y=x_hat_pf[:, i],
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
        showlegend=True
    ))
    
    # Set legend position: top right for x1, bottom right for x2
    legend_y = 0.98 if i == 0 else 0.02
    legend_yanchor = 'top' if i == 0 else 'bottom'
    
    fig.update_layout(
        title={
            'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Oscilador de Van der Pol',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title=state_info['y_label'],
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=legend_y,
            xanchor='right',
            yanchor=legend_yanchor,
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig.show()

# Gráfica de errores combinada
# error_x1_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
# error_x2_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_x1_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_x2_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
error_x1_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_x2_pf = x_true[:, 1] - x_hat_pf[:, 1]

fig_err = go.Figure()

# # Errores x1
# fig_err.add_trace(go.Scatter(
#     x=t_history, y=error_x1_ekf,
#     mode='lines',
#     name=f'EKF Error x₁ (MSE={mse_x1_ekf:.2e})',
#     line=dict(color='#1f77b4', width=1.5),
#     opacity=0.8
# ))

fig_err.add_trace(go.Scatter(
    x=t_history, y=error_x1_ukf,
    mode='lines',
    name=f'UKF Error x₁ (MSE={mse_x1_ukf:.2e})',
    line=dict(color='#2ca02c', width=1.5),
    opacity=0.8
))

fig_err.add_trace(go.Scatter(
    x=t_history, y=error_x1_pf,
    mode='lines',
    name=f'PF Error x₁ (MSE={mse_x1_pf:.2e})',
    line=dict(color='#d62728', width=1.5),
    opacity=0.8
))

# # Errores x2
# fig_err.add_trace(go.Scatter(
#     x=t_history, y=error_x2_ekf,
#     mode='lines',
#     name=f'EKF Error x₂ (MSE={mse_x2_ekf:.2e})',
#     line=dict(color='#1f77b4', width=1.5, dash='dot'),
#     opacity=0.8
# ))

fig_err.add_trace(go.Scatter(
    x=t_history, y=error_x2_ukf,
    mode='lines',
    name=f'UKF Error x₂ (MSE={mse_x2_ukf:.2e})',
    line=dict(color='#2ca02c', width=1.5, dash='dot'),
    opacity=0.8
))

fig_err.add_trace(go.Scatter(
    x=t_history, y=error_x2_pf,
    mode='lines',
    name=f'PF Error x₂ (MSE={mse_x2_pf:.2e})',
    line=dict(color='#d62728', width=1.5, dash='dot'),
    opacity=0.8
))

fig_err.update_layout(
    title={
        'text': 'Errores de Estimación - Oscilador de Van der Pol',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo (s)',
    yaxis_title='Error de Estimación',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.98,
        y=0.98,
        xanchor='right',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_err.show()

# Espacio de fase (Ciclo Límite)
fig_phase = go.Figure()

fig_phase.add_trace(go.Scatter(
    x=x_true[:, 0], y=x_true[:, 1],
    mode='lines',
    name='Ciclo Límite Real',
    line=dict(color='#000000', width=3)
))

# fig_phase.add_trace(go.Scatter(
#     x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1],
#     mode='lines',
#     name='Estimación EKF-RHONN',
#     line=dict(color='#1f77b4', width=2, dash='dash')
# ))

fig_phase.add_trace(go.Scatter(
    x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1],
    mode='lines',
    name='Estimación UKF-RHONN',
    line=dict(color='#2ca02c', width=2, dash='dot')
))

fig_phase.add_trace(go.Scatter(
    x=x_hat_pf[:, 0], y=x_hat_pf[:, 1],
    mode='lines',
    name='Estimación PF-RHONN',
    line=dict(color='#d62728', width=2, dash='dashdot')
))

fig_phase.update_layout(
    title={
        'text': 'Espacio de Fases - Oscilador de Van der Pol',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Posición x₁',
    yaxis_title='Velocidad x₂',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.98,
        y=0.02,
        xanchor='right',
        yanchor='bottom',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_width'],  # Aspecto cuadrado
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_phase.show()

# Gráfica de barras comparando MSE
fig_mse = go.Figure()

filters = ['UKF-RHONN', 'PF-RHONN']  # EKF commented out

fig_mse.add_trace(go.Bar(
    name='Posición x₁',
    x=filters,
    y=[mse_x1_ukf, mse_x1_pf],
    marker_color='#636EFA',
    text=[f'{mse_x1_ukf:.2e}', f'{mse_x1_pf:.2e}'],
    textposition='outside'
))

fig_mse.add_trace(go.Bar(
    name='Velocidad x₂',
    x=filters,
    y=[mse_x2_ukf, mse_x2_pf],
    marker_color='#EF553B',
    text=[f'{mse_x2_ukf:.2e}', f'{mse_x2_pf:.2e}'],
    textposition='outside'
))

fig_mse.update_layout(
    title={
        'text': 'Comparación de Error Cuadrático Medio (MSE)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tipo de Filtro',
    yaxis_title='Error Cuadrático Medio (MSE)',
    yaxis=dict(
        type='log',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    xaxis=dict(
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.98,
        y=0.98,
        xanchor='right',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    barmode='group',
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_mse.show()


In [14]:
# ============================================================
# 7) RMSE Calculation and Plots
# ============================================================

# Calculate RMSE for each filter and state
rmse_x1_ukf = np.sqrt(mse_x1_ukf)
rmse_x2_ukf = np.sqrt(mse_x2_ukf)
rmse_x1_pf = np.sqrt(mse_x1_pf)
rmse_x2_pf = np.sqrt(mse_x2_pf)

# Total RMSE (combined metric)
rmse_total_ukf = np.sqrt(mse_total_ukf)
rmse_total_pf = np.sqrt(mse_total_pf)

# Print RMSE comparison
print("\n" + "="*70)
print("📊 ROOT MEAN SQUARE ERROR (RMSE) COMPARISON")
print("="*70)
print("\nUKF-RHONN:")
print(f"  RMSE x₁ (posición):  {rmse_x1_ukf:.6f}")
print(f"  RMSE x₂ (velocidad): {rmse_x2_ukf:.6f}")
print(f"  RMSE Total:          {rmse_total_ukf:.6f}")

print("\nPF-RHONN:")
print(f"  RMSE x₁ (posición):  {rmse_x1_pf:.6f}")
print(f"  RMSE x₂ (velocidad): {rmse_x2_pf:.6f}")
print(f"  RMSE Total:          {rmse_total_pf:.6f}")

print("\n" + "="*70)
print(f"🏆 MEJOR FILTRO (RMSE): ", end="")
rmse_dict = {'UKF': rmse_total_ukf, 'PF': rmse_total_pf}
best_filter_rmse = min(rmse_dict, key=rmse_dict.get)
print(f"{best_filter_rmse} (RMSE total: {rmse_dict[best_filter_rmse]:.6f})")
print("="*70)

# Bar chart comparing RMSE
fig_rmse = go.Figure()

filters = ['UKF-RHONN', 'PF-RHONN']

fig_rmse.add_trace(go.Bar(
    name='Posición x₁',
    x=filters,
    y=[rmse_x1_ukf, rmse_x1_pf],
    marker_color='#636EFA',
    text=[f'{rmse_x1_ukf:.4f}', f'{rmse_x1_pf:.4f}'],
    textposition='outside'
))

fig_rmse.add_trace(go.Bar(
    name='Velocidad x₂',
    x=filters,
    y=[rmse_x2_ukf, rmse_x2_pf],
    marker_color='#EF553B',
    text=[f'{rmse_x2_ukf:.4f}', f'{rmse_x2_pf:.4f}'],
    textposition='outside'
))

fig_rmse.add_trace(go.Bar(
    name='RMSE Total',
    x=filters,
    y=[rmse_total_ukf, rmse_total_pf],
    marker_color='#00CC96',
    text=[f'{rmse_total_ukf:.4f}', f'{rmse_total_pf:.4f}'],
    textposition='outside'
))

fig_rmse.update_layout(
    title={
        'text': 'Comparación de Error Cuadrático Medio Raíz (RMSE)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tipo de Filtro',
    yaxis_title='Error Cuadrático Medio Raíz (RMSE)',
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    xaxis=dict(
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.98,
        y=0.98,
        xanchor='right',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    barmode='group',
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_rmse.show()

# Time-varying RMSE (cumulative moving window)
window_size = 50  # 50 time steps moving window

rmse_time_x1_ukf = np.zeros(n_steps - window_size)
rmse_time_x2_ukf = np.zeros(n_steps - window_size)
rmse_time_x1_pf = np.zeros(n_steps - window_size)
rmse_time_x2_pf = np.zeros(n_steps - window_size)

for i in range(n_steps - window_size):
    rmse_time_x1_ukf[i] = np.sqrt(np.mean((x_true[i:i+window_size, 0] - x_hat_ukf[i:i+window_size, 0])**2))
    rmse_time_x2_ukf[i] = np.sqrt(np.mean((x_true[i:i+window_size, 1] - x_hat_ukf[i:i+window_size, 1])**2))
    rmse_time_x1_pf[i] = np.sqrt(np.mean((x_true[i:i+window_size, 0] - x_hat_pf[i:i+window_size, 0])**2))
    rmse_time_x2_pf[i] = np.sqrt(np.mean((x_true[i:i+window_size, 1] - x_hat_pf[i:i+window_size, 1])**2))

t_rmse = t_history[window_size:]

# Plot time-varying RMSE
fig_rmse_time = go.Figure()

fig_rmse_time.add_trace(go.Scatter(
    x=t_rmse, y=rmse_time_x1_ukf,
    mode='lines',
    name='UKF RMSE x₁',
    line=dict(color='#2ca02c', width=2)
))

fig_rmse_time.add_trace(go.Scatter(
    x=t_rmse, y=rmse_time_x1_pf,
    mode='lines',
    name='PF RMSE x₁',
    line=dict(color='#d62728', width=2)
))

fig_rmse_time.add_trace(go.Scatter(
    x=t_rmse, y=rmse_time_x2_ukf,
    mode='lines',
    name='UKF RMSE x₂',
    line=dict(color='#2ca02c', width=2, dash='dot')
))

fig_rmse_time.add_trace(go.Scatter(
    x=t_rmse, y=rmse_time_x2_pf,
    mode='lines',
    name='PF RMSE x₂',
    line=dict(color='#d62728', width=2, dash='dot')
))

fig_rmse_time.update_layout(
    title={
        'text': f'RMSE en Ventana Móvil (ventana={window_size} pasos)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo (s)',
    yaxis_title='RMSE en Ventana Móvil',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.98,
        y=0.98,
        xanchor='right',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_rmse_time.show()


📊 ROOT MEAN SQUARE ERROR (RMSE) COMPARISON

UKF-RHONN:
  RMSE x₁ (posición):  0.007664
  RMSE x₂ (velocidad): 0.007049
  RMSE Total:          0.010412

PF-RHONN:
  RMSE x₁ (posición):  0.001275
  RMSE x₂ (velocidad): 0.001586
  RMSE Total:          0.002035

🏆 MEJOR FILTRO (RMSE): PF (RMSE total: 0.002035)
